# 03 - Lakebase serving layer + Genie
* Lakebase project `support-command-center`, branch `production`, database `databricks_postgres`
* Snapshot synced tables (UC -> Postgres schema `support_command_center`): `lb_service_health_region_5m`, `lb_ticket_anomalies`, `lb_incident_scorecard`, `lb_region_heatmap_5m`, `lb_frustration_index_5m` - low-latency reads for the app
* App-owned OLTP tables (incident actions, alert acks, known-issue banners, demo clock) are created by the app service principal in schema `command_center_ops` on first start
* Genie space: **Fanimation Support Command Center** (`01f1d226b70f1202b656c554ffc9bd38`)

Re-run the sync cell after re-running notebook 02 (snapshot mode).

In [0]:
%pip install -q --upgrade "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import SyncedTable, SyncedTableSyncedTableSpec, SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy as P
w=WorkspaceClient()
FQ="classic_stable_l0m1dn_catalog.support_command_center"
specs={"gold_service_health_region_5m":["ts","region_group","domain"], "gold_ticket_anomalies":["anomaly_id"], "gold_incident_scorecard":["incident_id"],
       "gold_region_heatmap_5m":["ts","country_code"], "gold_frustration_index_5m":["scope_type","scope","ts"]}
for src,pk in specs.items():
    tgt=f"{FQ}.lb_{src.replace('gold_','')}"
    try:
        st=w.postgres.get_synced_table(name=f"synced_tables/{tgt}"); print("exists", tgt, st.status.detailed_state)
    except Exception:
        w.postgres.create_synced_table(synced_table=SyncedTable(spec=SyncedTableSyncedTableSpec(
            source_table_full_name=f"{FQ}.{src}", branch="projects/support-command-center/branches/production",
            primary_key_columns=pk, scheduling_policy=P.SNAPSHOT, postgres_database="databricks_postgres",
            create_database_objects_if_missing=True)), synced_table_id=tgt)
        print("created", tgt)